# Realistic scenario: sKF (Gaussian) vs sKF-L (minorized and exact) under heavy-tailed noise

Three closed-form filters on the setup of section 7 of the draft: an $M = 128$ room impulse
response, correlated input, heavy-tailed noise. No numerical integration anywhere. Metric:
misalignment $10\log_{10}\left(\|w_t - h_o\|^2/\|h_o\|^2\right)$, averaged over realisations.

## 1. Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr, logsumexp
from scipy.stats import gennorm
import rir_generator as rir

NOTEBOOK_START = time.time()                   # for the total printed at the very end
print("imports ready")

## 2. The scenario

* **Room:** $h_o$ = first $M = 128$ samples of the $(5, 10, 6)$ m room of section 7.1, $T_{60} = 200$ ms, 8 kHz. Fixed.
* **Input:** $x_t = -0.9\,x_{t-1} + u_t$, unit variance. **Observation:** $d_t = x_t^{\mathsf T}h_o + \eta_t$.
* **Noise:** generalized Gaussian, $\beta^* = 0.2$, at SNR $= 5$ dB, as in Fig. 3 of the paper.
* **What the filters assume:** Gaussian noise of variance $v_\eta$, or Laplacian of scale $b_\eta = \sqrt{v_\eta/2}$.
* **Run:** $N = 96000$ steps, 12 s at 8 kHz, and $R = 20$ realisations.

In [ ]:
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run (12 s at 8 kHz, the span of Fig. 3a of the paper)
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary

AR_A = -0.9                 # AR(1) coefficient of the input
SNR_DB = 5.0                # nominal signal-to-noise ratio, as in Fig. 3 of the paper
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
VAR_THETA_0 = 2.0           # initial prior variance on each weight, as in notebooks 01 and 02
FS = 8000                   # sampling rate [Hz], the rate of section 7.1 of the draft

# The room of section 7.1 of the draft, simulated with ref. [24] (image method).
ROOM = [5, 10, 6]           # room dimensions [m]
T60 = 0.2                   # reverberation time [s]
C_SOUND = 340               # speed of sound [m/s]
SRC = [1, 2.5, 2]           # source position [m]
MIC = [1, 1.5, 1]           # receiver position [m]

# Generated once and then fixed. Normalised to unit norm: misalignment is scale-invariant
# and SNR_DB is imposed below from P_signal, so the normalisation changes nothing except
# keeping the hand-tuned epsilon of section 4 on the same scale it was tuned at.
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM,
                  reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)

# Signal power, in closed form: r[k] = AR_A^|k| for a unit-variance AR(1) input.
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))
Rxx = AR_A**lags
P_signal = ho @ Rxx @ ho

var_eta = P_signal/10**(SNR_DB/10)                 # noise variance that gives SNR_DB
b_eta = np.sqrt(var_eta/2)                         # Laplacian scale with that same variance

# Scale of the generalized Gaussian with variance var_eta:
# Var = scale^2 Gamma(3/beta)/Gamma(1/beta).
scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))

print(f"signal power = {P_signal:.4f}")
print(f"var_eta      = {var_eta:.4e}   (nominal, at {SNR_DB:.0f} dB SNR)")
print(f"b_eta        = {b_eta:.4e}")
print(f"noise scale  = {scale_gg:.4e}   (generalized Gaussian, beta = {BETA})")

`generate_signals(seed)` returns one realisation: the input `x`, the observation `d` and the noise `eta`.

In [ ]:
def generate_signals(seed):
    """One realisation: AR(1) input, its clean output through ho, and heavy-tailed noise."""
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - AR_A**2)*rng.standard_normal(N + WARMUP)   # driving noise, unit-variance x
    x = lfilter([1.0], [1.0, -AR_A], u)[WARMUP:]               # x_t = AR_A x_{t-1} + u_t
    y = np.convolve(ho, x)[:N]                                 # clean output, y_t = x_t' ho
    eta = gennorm.rvs(BETA, scale=scale_gg, size=N, random_state=rng)
    return x, y + eta, eta


x, d, eta = generate_signals(0)
print("x:", x.shape, " d:", d.shape)
print(f"noise, first realisation: std {eta.std():.4f}, "
      f"median |eta| {np.median(np.abs(eta)):.5f}, max |eta| {np.abs(eta).max():.4f}")

**Figure 1.** One realisation of the noise. Dashed: the nominal $\pm\sqrt{v_\eta}$ both filters were told to expect.

In [ ]:
x, d, eta = generate_signals(0)

plt.figure(figsize=(10, 4))
plt.plot(eta, linewidth=0.6)
plt.axhline(np.sqrt(var_eta), color="k", linestyle="--", linewidth=0.8,
            label=r"nominal $\pm\sqrt{v_\eta}$")
plt.axhline(-np.sqrt(var_eta), color="k", linestyle="--", linewidth=0.8)
plt.xlabel("time step $t$")
plt.ylabel(r"$\eta_t$")
plt.title(f"Measurement noise, one realisation (generalized Gaussian, beta = {BETA})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

median_eta = np.median(np.abs(eta))
print(f"median |eta| = {median_eta:.5f}")
print(f"nominal std  = {np.sqrt(var_eta):.5f}   ({np.sqrt(var_eta)/median_eta:.0f}x the median)")
print(f"max |eta|    = {np.abs(eta).max():.4f}   ({np.abs(eta).max()/median_eta:.0f}x the median)")

## 3. The three closed forms

Copied unchanged: the Gaussian and exact ones from notebooks 01 and 02, the minorized one from Ramiro.
Notation: $e_t = d_t - x_t^{\mathsf T}w_{t-1}$, $\tilde v_t$ predicted variance, $\varepsilon$ process noise, $M$ weights.

### sKF (Gaussian), eq. (32) of the published paper

The paper's $w_t = w_{t-1} + \kappa_t\alpha_t e_t$, with $\bar V_t = \tilde v_t I$ and $h_t = 1/v_\eta$:

$$\tilde v_t = v_{t-1} + \varepsilon \qquad \alpha_t = \frac{\tilde v_t}{v_\eta + \tilde v_t\|x_t\|^2}$$

$$w_t = w_{t-1} + \alpha_t\,x_t\,e_t \qquad v_t = \tilde v_t\left(1 - \frac{\alpha_t\|x_t\|^2}{M}\right)$$

In [ ]:
def shift(new_x_sample, x_window):
    L = len(x_window)
    new_x_window = np.zeros(L)
    new_x_window[0] = new_x_sample
    new_x_window[1:] = x_window[:-1]
    return new_x_window


def sKF_closed_form(N, x, d, w0, parameters):
    """Closed form from the published paper (eq. 32): the same filter, solved by hand."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    var_eta = parameters["var_eta"]                    # v_eta, observation noise variance
    L = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(L)                                  # x_t, window with the last L samples
    w_hist = np.zeros((N, L))                          # weights at every step
    var_hist = np.zeros((N,))                          # variance at every step
    e = np.zeros((N,))                                 # e_t, prediction error at every step

    for k in range(N):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does
        var_hist[k] = v                                # same for the variance

        if k >= L:                                     # Augusto waits for a full window; same here
            v_pred = v + epsilon                       # v_tilde = v + eps
            power = x_t @ x_t                          # ||x_t||^2
            gain = v_pred/(var_eta + v_pred*power)     # alpha = v_tilde / (v_eta + v_tilde ||x||^2)
            w = w + gain*x_t*e[k]                      # w = w + alpha x_t e_t
            v = v_pred*(1 - gain*power/L)              # v = v_tilde (1 - alpha ||x||^2 / L)

    return {"h": w, "e": e, "w_hist": w_hist, "var_hist": var_hist}


print("Gaussian closed form ready")

### sKF-L (minorized), section 4.1 of the draft

The Gaussian filter with $v_\eta \leftarrow b_\eta|e_t|$. Ramiro's implementation.

$$w_t = w_{t-1} + \frac{\tilde v_t\,x_t}{b_\eta|e_t| + \tilde v_t\|x_t\|^2}\,e_t$$

$$v_t = \tilde v_t\left(1 - \frac{1}{M}\frac{\tilde v_t\|x_t\|^2}{b_\eta|e_t| + \tilde v_t\|x_t\|^2}\right)$$

In [ ]:
# === RAMIRO: sKF-L minorized (draft eqs. 50 and 51) - copied verbatim, NOT edited ===
# source: https://github.com/Ramirouf/INRS-bayes-adaptive-filters
#         bayes-adaptive-filters.ipynb, cell "Adaptive Filters", commit dbab124
#
# Its interface differs from the two filters above: the initial variance is read from
# parameters["v_tilde_0"], and the weight history comes back under the key "h". The call
# site adapts to that; the function is left exactly as it is. See Observations.
def sKF_L_algorithm(N, x, d, h0, parameters):
    h = h0
    epsilon = parameters["epsilon"]
    b_eta = parameters["b_eta"]
    v_tilde_0 = parameters["v_tilde_0"]

    # normalize x
    # regularization = 1e-3
    # x_reg = np.sign(x) * (np.abs(x) + regularization)

    L = len(h)
    y = np.zeros((N,))
    e = np.zeros((N,))
    xtemp = np.zeros(L)
    v=v_tilde_0
    h_hist = np.zeros((N, L))
    v_hist = np.zeros((N, L))
    # d: salida del sistema con ruido
    # y: salida estimada
    for k in range(0,N):
        xtemp = shift(x[k], xtemp)
        y[k] = h @ xtemp
        e[k] = d[k] - y[k]
        h_hist[k] = h
        v_hist[k] = v

        if k >= L:
            # predict
            v += epsilon
            # update
            norm = xtemp @ xtemp
            s = b_eta * abs(e[k]) + v * norm
            h = h + xtemp * (v * e[k]/s) # not h+= because it would mutate h0
            v = v * (1 - (v * norm) / (L * s)) 

    return {'h': h_hist, 'y': y, 'e': e, 'v': v_hist}
print("minorized closed form ready (Ramiro's implementation)")

### sKF-L (exact), section 5 of the draft

Same prediction. With $\varsigma = \pm 1$, and $\phi$, $\Phi$ the standard normal pdf and cdf:

$$\kappa_\varsigma = \frac{\varsigma e_t - \tilde v_t\|x_t\|^2/b_\eta}{\sqrt{\tilde v_t}\,\|x_t\|}
\qquad \pi_\varsigma = \mathrm{softmax}\!\left(-\varsigma e_t/b_\eta + \log\Phi(\kappa_\varsigma)\right)
\qquad h(\kappa) = \phi(\kappa)/\Phi(\kappa)$$

$$\Lambda = \pi_+ - \pi_- \qquad \Gamma = \pi_+h(\kappa_+) - \pi_-h(\kappa_-) \qquad
P = \pi_+h(\kappa_+) + \pi_-h(\kappa_-) \qquad Q = \pi_+\kappa_+h(\kappa_+) + \pi_-\kappa_-h(\kappa_-)$$

$$w_t = w_{t-1} + \left(\frac{\tilde v_t}{b_\eta}\Lambda - \frac{\sqrt{\tilde v_t}}{\|x_t\|}\Gamma\right)x_t$$

$$D_t = \gamma^2(1-\Lambda^2) - 2\gamma\lambda(P - \Lambda\Gamma) - \lambda^2(Q + \Gamma^2)
\qquad v_t = \tilde v_t + \frac{1}{M}D_t\|x_t\|^2 \qquad \gamma = \frac{\tilde v_t}{b_\eta},\ \lambda = \frac{\sqrt{\tilde v_t}}{\|x_t\|}$$

**The three differ only in the correction:** Gaussian $\alpha_t e_t$, unbounded; minorized, the same with $v_\eta \to b_\eta|e_t|$; exact, $\Lambda \in [-1, 1]$.

In [ ]:
SIGN = np.array([1.0, -1.0])                           # the two branches, varsigma = +1 and -1


def sKF_L_closed_form(N, x, d, w0, parameters):
    """Closed form of section 5 (sKF-L, exact): the same filter, solved by hand."""
    epsilon = parameters["epsilon"]                    # eps, process noise variance
    b_eta = parameters["b_eta"]                        # b_eta, Laplacian observation scale
    M = len(w0)                                        # number of weights
    w = w0.copy()                                      # w, current weights, start at w0
    v = parameters["var_theta_0"]                      # v, scalar prior variance on each weight
    x_t = np.zeros(M)                                  # x_t, window with the last M samples
    w_hist = np.zeros((N, M))                          # weights at every step
    var_hist = np.zeros((N,))                          # variance at every step
    e = np.zeros((N,))                                 # e_t, prediction error at every step

    for k in range(N):
        x_t = shift(x[k], x_t)                         # new sample in, window slides
        e[k] = d[k] - x_t @ w                          # e_t = d_t - x_t' w_{t-1}

        w_hist[k] = w.copy()                           # state BEFORE the update, as Augusto does
        var_hist[k] = v                                # same for the variance

        if k >= M:                                     # Augusto waits for a full window; same here
            v_tilde = v + epsilon                      # v_tilde = v + eps
            norm_x = np.sqrt(x_t @ x_t)                # ||x_t||
            power = norm_x**2                          # ||x_t||^2

            # The two mixture arguments. Global: no dependence on the weight index m.
            kappa = (SIGN*e[k] - v_tilde*power/b_eta)/(np.sqrt(v_tilde)*norm_x)

            # Mixture weights, in the log domain. Phi(kappa) underflows and e^{2 e / b_eta}
            # overflows, so neither factor may be formed on its own.
            log_Phi = log_ndtr(kappa)                  # log Phi(kappa), accurate in the left tail
            log_pi = -SIGN*e[k]/b_eta + log_Phi
            pi = np.exp(log_pi - logsumexp(log_pi))    # softmax, sums to 1

            # Inverse Mills ratio h = phi/Phi, also in the log domain.
            log_phi = -0.5*kappa**2 - 0.5*np.log(2*np.pi)
            h = np.exp(log_phi - log_Phi)

            Lambda = np.sum(SIGN*pi)                   # saturating prediction error, in [-1, 1]
            Gamma = np.sum(SIGN*pi*h)
            P = np.sum(pi*h)
            Q = np.sum(pi*kappa*h)

            gain = v_tilde*Lambda/b_eta - np.sqrt(v_tilde)*Gamma/norm_x
            w = w + gain*x_t                           # w = w + (v~ L/b_eta - sqrt(v~) G/||x||) x

            gamma = v_tilde/b_eta
            lam = np.sqrt(v_tilde)/norm_x
            D = (gamma**2*(1 - Lambda**2)
                 - 2*gamma*lam*(P - Lambda*Gamma)
                 - lam**2*(Q + Gamma**2))
            v = v_tilde + D*power/M                    # v = v~ + D ||x||^2 / M

    return {"h": w, "e": e, "w_hist": w_hist, "var_hist": var_hist}


print("Laplacian closed form ready")

In [ ]:
# Convergence criterion, used both by the run below and by the grid search of section 4.
# It reads the tail off the array itself so that it works at the full N and at the shorter
# lengths the search uses. The rule is unchanged: floor = mean over the last quarter of the
# run, converged = first step within 3 dB of that floor.
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


print("convergence criterion ready")

## 4. Choosing $\varepsilon$, and the run

**Target first, parameters after**, as in the paper: $\bar{\mathsf m}_\infty = -20$ dB, the value its Fig. 3 pairs with SNR $= 5$ dB.
Grid search, the paper's protocol: scan a log grid of $\varepsilon$ per filter and keep the value whose floor lands on the target.
The search runs cheap, $R = 3$ and shorter runs; the three selected $\varepsilon$ then run at $R = 20$ and $N = 96000$.

In [ ]:
R_SEARCH = 3                # realisations while scanning the grid
R_PLOT = 10                 # realisations for the runs plotted and tabulated in section 6
N_SEARCH = 24000            # scan length for the two Laplacian filters, which converge fast
N_SEARCH_GAUSS = 96000      # the Gaussian filter converges ~10x slower, so it scans at full length
TARGET_DB = -20.0           # target misalignment

EPS_GRID_LAPLACIAN = np.logspace(-8, -3, 11)
EPS_GRID_GAUSS = np.logspace(-11, -6, 11)

for label, grid in [("sKF-L, minorized and exact", EPS_GRID_LAPLACIAN),
                    ("sKF, Gaussian", EPS_GRID_GAUSS)]:
    print(f"{label}: {len(grid)} values of epsilon over {np.log10(grid[-1]/grid[0]):.0f} decades")
    print("   " + "  ".join(f"{epsilon:.1e}" for epsilon in grid))

w0 = np.zeros(M)                               # all three filters start from zero
energy_ho = ho @ ho                            # ||ho||^2, the denominator of misalignment

# One set of signals shared by every filter and every epsilon, so the comparison is paired.
search_signals = [generate_signals(seed)[:2] for seed in range(R_SEARCH)]

In [ ]:
def run_filter(name, epsilon, n, signals):
    """Average misalignment curve of one filter at one epsilon, over the given signals."""
    misalignment = np.zeros(n)
    for x, d in signals:
        if name == "sKF (Gaussian)":
            p = {"epsilon": epsilon, "var_eta": var_eta, "var_theta_0": VAR_THETA_0}
            w = sKF_closed_form(n, x[:n], d[:n], w0, p)["w_hist"]
        elif name == "sKF-L (minorized)":
            p = {"epsilon": epsilon, "b_eta": b_eta, "v_tilde_0": VAR_THETA_0}   # Ramiro's key names
            w = sKF_L_algorithm(n, x[:n], d[:n], w0, p)["h"]
        else:
            p = {"epsilon": epsilon, "b_eta": b_eta, "var_theta_0": VAR_THETA_0}
            w = sKF_L_closed_form(n, x[:n], d[:n], w0, p)["w_hist"]
        misalignment += ((w - ho)**2).sum(axis=1)/energy_ho
    return misalignment/len(signals)


def scan_grid(name, eps_grid, n, signals):
    """Floor and steps to converge at every epsilon of the grid. The whole grid, no bisection: see section 9."""
    results = [steady_state(run_filter(name, e, n, signals)) for e in eps_grid]
    return np.array([floor for floor, _ in results]), np.array([steps for _, steps in results])


def pick_epsilon(name, eps_grid, floors, target, n, signals):
    """Interpolate the epsilon that lands on the target, run it once on `signals`, report it."""
    best = int(np.argmin(floors))
    branch_eps, branch_floors = eps_grid[best:], floors[best:]   # below the optimum it stalls

    if target > branch_floors.max():
        return None, branch_floors.max(), None, "grid too narrow", None

    if target < branch_floors.min():
        epsilon, status = branch_eps[0], "target not reached"      # report its best floor instead
    else:
        order = np.argsort(branch_floors)
        epsilon = 10**np.interp(target, branch_floors[order], np.log10(branch_eps)[order])
        status = "ok" if best > 0 else "ok (optimum at grid edge)"

    misalignment = run_filter(name, epsilon, n, signals)
    floor, reached = steady_state(misalignment)
    return epsilon, floor, reached, status, misalignment


print("search functions ready")

In [ ]:
SEARCHES = [("sKF (Gaussian)", EPS_GRID_GAUSS, N_SEARCH_GAUSS),
            ("sKF-L (minorized)", EPS_GRID_LAPLACIAN, N_SEARCH),
            ("sKF-L (exact)", EPS_GRID_LAPLACIAN, N_SEARCH)]

start = time.time()
scans = {name: scan_grid(name, grid, n, search_signals) for name, grid, n in SEARCHES}   # (floors, steps)
selected = {name: pick_epsilon(name, grid, scans[name][0], TARGET_DB, n, search_signals)
            for name, grid, n in SEARCHES}

print(f"grid search at target {TARGET_DB:.0f} dB, R = {R_SEARCH}\n")
print(f"{'filter':<20}{'epsilon':>11}{'floor [dB]':>12}{'steps':>9}  status")
for name, (epsilon, floor, reached, status, _) in selected.items():
    eps_text = "-" if epsilon is None else f"{epsilon:.2e}"
    steps_text = "-" if reached is None else str(reached)
    print(f"{name:<20}{eps_text:>11}{floor:>12.2f}{steps_text:>9}  {status}")

EPSILON_GAUSS = selected["sKF (Gaussian)"][0]
EPSILON_MIN = selected["sKF-L (minorized)"][0]
EPSILON_L = selected["sKF-L (exact)"][0]
print(f"\nphase 1, grid search: {time.time() - start:.0f} s")

In [ ]:
# EPSILON_GAUSS, EPSILON_MIN and EPSILON_L come from the grid search above.
parameters_gauss = {"epsilon": EPSILON_GAUSS, "var_eta": var_eta, "var_theta_0": VAR_THETA_0}
parameters_min = {"epsilon": EPSILON_MIN, "b_eta": b_eta, "v_tilde_0": VAR_THETA_0}
parameters_L = {"epsilon": EPSILON_L, "b_eta": b_eta, "var_theta_0": VAR_THETA_0}

misalignment_gauss = np.zeros(N)                   # running sums over the realisations
misalignment_min = np.zeros(N)
misalignment_L = np.zeros(N)

start = time.time()
for realisation in range(R):
    x, d, eta = generate_signals(realisation)

    w_gauss = sKF_closed_form(N, x, d, w0, parameters_gauss)["w_hist"]
    w_min = sKF_L_algorithm(N, x, d, w0, parameters_min)["h"]   # Ramiro returns it under "h"
    w_L = sKF_L_closed_form(N, x, d, w0, parameters_L)["w_hist"]

    misalignment_gauss += ((w_gauss - ho)**2).sum(axis=1)/energy_ho
    misalignment_min += ((w_min - ho)**2).sum(axis=1)/energy_ho
    misalignment_L += ((w_L - ho)**2).sum(axis=1)/energy_ho

misalignment_gauss /= R                            # ensemble average
misalignment_min /= R
misalignment_L /= R

print(f"{R} realisations of {N} steps, done in {time.time() - start:.0f} s")

## 5. Results of the run

**Figure 2.** The Gaussian filter on the left, the two Laplacian ones on the right, as in Fig. 3 of the paper.
Same $y$ axis; independent $x$ axes, because the difference in time scale is the point.

In [ ]:
db_gauss = 10*np.log10(misalignment_gauss)
db_min = 10*np.log10(misalignment_min)
db_L = 10*np.log10(misalignment_L)
t_axis = np.arange(N)/FS                           # steps to seconds, as in Fig. 3 of the paper

T_ROBUST = 2.0                                     # seconds shown in (b): they settle well before
y_lim = (min(db_gauss.min(), db_min.min(), db_L.min()) - 1, 1)

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_a.plot(t_axis, db_gauss, color="C0",
          label=f"sKF (Gaussian), eps = {EPSILON_GAUSS:.1e}", linewidth=1.2)
ax_a.set_title("(a) conventional, beta = 2.0")
ax_a.set_xlim(0, N/FS)

ax_b.plot(t_axis, db_min, color="C1",
          label=f"sKF-L (minorized), eps = {EPSILON_MIN:.1e}", linewidth=1.2)
ax_b.plot(t_axis, db_L, color="C2",
          label=f"sKF-L (exact), eps = {EPSILON_L:.1e}", linewidth=1.2)
ax_b.set_title("(b) robust, beta = 1.0")
ax_b.set_xlim(0, T_ROBUST)

for ax in (ax_a, ax_b):
    ax.set_xlabel("$t$ [sec]")
    ax.set_ylabel("misalignment [dB]")
    ax.set_ylim(*y_lim)
    ticks = [t for t in ax.get_yticks() if y_lim[0] <= t <= y_lim[1]]
    ax.set_yticks(sorted(set(ticks) | {TARGET_DB}))       # the -20 dB level, readable off the axis
    ax.legend()
    ax.grid(alpha=0.3)

fig.suptitle(f"Convergence in heavy-tailed noise (beta* = {BETA}), SNR = {SNR_DB:.0f} dB, "
             f"average of {R} realisations")
plt.tight_layout()
plt.show()

The parameters behind Figure 2, in the spirit of Table 1 of the paper, then the floors and speed-ups.
Floor: mean of the last quarter of the run. Converged: first step within 3 dB of that floor.

In [ ]:
tail = slice(3*N//4, N)                            # last quarter of the run

table = [("sKF (Gaussian)", EPSILON_GAUSS, misalignment_gauss),
         ("sKF-L (minorized)", EPSILON_MIN, misalignment_min),
         ("sKF-L (exact)", EPSILON_L, misalignment_L)]

print(f"{'filter':<20}{'epsilon':>12}{'floor [dB]':>14}")
for name, epsilon, misalignment in table:
    print(f"{name:<20}{epsilon:>12.1e}{10*np.log10(misalignment[tail].mean()):>14.2f}")

In [ ]:
tail = slice(3*N//4, N)


floor_gauss, reached_gauss = steady_state(misalignment_gauss)
floor_min, reached_min = steady_state(misalignment_min)
floor_L, reached_L = steady_state(misalignment_L)

print(f"sKF (Gaussian),    eps = {EPSILON_GAUSS:.1e}")
print(f"   steady-state misalignment : {floor_gauss:+.2f} dB")
print(f"   steps to floor + 3 dB     : {reached_gauss}")
print(f"sKF-L (minorized), eps = {EPSILON_MIN:.1e}")
print(f"   steady-state misalignment : {floor_min:+.2f} dB")
print(f"   steps to floor + 3 dB     : {reached_min}")
print(f"sKF-L (exact),     eps = {EPSILON_L:.1e}")
print(f"   steady-state misalignment : {floor_L:+.2f} dB")
print(f"   steps to floor + 3 dB     : {reached_L}")
print()
print(f"spread of the three floors      : {max(floor_gauss, floor_min, floor_L) - min(floor_gauss, floor_min, floor_L):.2f} dB")
print(f"exact vs Gaussian, speed-up     : {reached_gauss/reached_L:.2f}x")
print(f"exact vs minorized, speed-up    : {reached_min/reached_L:.2f}x")
print(f"exact vs minorized, floor gap   : {floor_L - floor_min:+.2f} dB")

## 6. Does the minorized filter's advantage survive?

As $\varepsilon$ varies, each filter traces a curve of floor against steps to converge. That curve is the filter; $\varepsilon$ only says where on it you stand.
If one curve lies below the other across the reachable range, that filter dominates, and no tuning protocol can be blamed for the result.
The curve is at SNR $= 5$ dB, both filters on the same signals, $R = 10$ and $N = 48000$. The SNR sweep after it covers other noise levels.

In [ ]:
LAPLACIAN = ["sKF-L (minorized)", "sKF-L (exact)"]
N_TRADEOFF = 48000          # longer than N_SEARCH: at small epsilon the filters settle slowly, and
                            # the floor is read off the last quarter of the run

# Both filters on the same R_PLOT realisations, as in the search, so the two curves are paired.
plot_signals = [generate_signals(seed)[:2] for seed in range(R_PLOT)]

start = time.time()
tradeoff = {name: scan_grid(name, EPS_GRID_LAPLACIAN, N_TRADEOFF, plot_signals) for name in LAPLACIAN}
print(f"phase 2, trade-off scan (R = {R_PLOT}, N = {N_TRADEOFF}): {time.time() - start:.0f} s")

**Same $\varepsilon$.** Both Laplacian filters at the $\varepsilon$ the search picked for the exact one, at full $R$ and $N$.
They are the two starred points of Figure 3.

In [ ]:
start = time.time()
parameters_same = {"epsilon": EPSILON_L, "b_eta": b_eta, "v_tilde_0": VAR_THETA_0}

misalignment_same = np.zeros(N)
for realisation in range(R):
    x, d, eta = generate_signals(realisation)
    w = sKF_L_algorithm(N, x, d, w0, parameters_same)["h"]
    misalignment_same += ((w - ho)**2).sum(axis=1)/energy_ho
misalignment_same /= R

floor_same, reached_same = steady_state(misalignment_same)
print(f"both Laplacian filters at eps = {EPSILON_L:.2e}")
print(f"   minorized : floor {floor_same:+.2f} dB, {reached_same} steps")
print(f"   exact     : floor {floor_L:+.2f} dB, {reached_L} steps")
print(f"   minorized relative to exact: {floor_same - floor_L:+.2f} dB of floor, "
      f"{100*(reached_same/reached_L - 1):+.1f}% steps")
print(f"   largest separation between the two curves: "
      f"{np.abs(10*np.log10(misalignment_same) - db_L).max():.2f} dB")
print(f"\nsame-epsilon run: {time.time() - start:.0f} s")

**Figure 3.** The trade-off curve: floor against steps to converge, one point per $\varepsilon$ of the grid. **Lower and further left is better.**
Stars: the same-$\varepsilon$ runs above, at $R = 20$ and $N = 96000$. Stalled and unsettled grid points are left out and listed in the table below.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
curves = {}                                    # the plotted points of each filter, for the crossing check
for name, color in [("sKF-L (minorized)", "C1"), ("sKF-L (exact)", "C2")]:
    floors, steps = tradeoff[name]
    best = int(np.argmin(floors))              # below its optimum the filter stalls (section 9)
    keep = (np.arange(len(floors)) >= best) & (steps <= N_TRADEOFF/2)   # and unsettled runs are out
    curves[name] = (floors[keep], steps[keep])
    ax.plot(floors[keep], steps[keep], "o-", color=color, label=name)

for label, floor, steps, color in [("minorized", floor_same, reached_same, "C1"),
                                   ("exact", floor_L, reached_L, "C2")]:
    ax.plot(floor, steps, "*", color=color, markersize=16, markeredgecolor="k")
    ax.annotate(f"{label}, eps = {EPSILON_L:.2e}", (floor, steps), textcoords="offset points",
                xytext=(8, 8), fontsize=9)

low, high = ax.get_xlim()
ax.set_xticks(np.arange(5*np.ceil(low/5), high, 5))      # every 5 dB, so -20 is a tick
ax.set_xlabel("achieved floor [dB]")
ax.set_ylabel("steps to converge")
ax.set_title(f"Trade-off at SNR = {SNR_DB:.0f} dB: lower and further left is better")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'filter':<20}{'epsilon':>10}{'floor [dB]':>12}{'steps':>8}  point")
for name in LAPLACIAN:
    floors, steps = tradeoff[name]
    best = int(np.argmin(floors))
    for i, (epsilon, floor, step) in enumerate(zip(EPS_GRID_LAPLACIAN, floors, steps)):
        point = "stalled" if i < best else "not settled" if step > N_TRADEOFF/2 else "plotted"
        print(f"{name:<20}{epsilon:>10.1e}{floor:>12.2f}{step:>8}  {point}")

# Do the curves cross? Steps of both at the same floor, over the floor range both curves cover.
(floors_min, steps_min), (floors_ex, steps_ex) = curves["sKF-L (minorized)"], curves["sKF-L (exact)"]
common = np.linspace(max(floors_min.min(), floors_ex.min()), min(floors_min.max(), floors_ex.max()), 200)
gap = (np.interp(common, np.sort(floors_ex), steps_ex[np.argsort(floors_ex)])
       - np.interp(common, np.sort(floors_min), steps_min[np.argsort(floors_min)]))   # exact minus minorized
crossings = common[1:][np.diff(np.sign(gap)) != 0]
print(f"\ncommon floor range: {common[0]:.1f} to {common[-1]:.1f} dB")
if crossings.size:
    print("the curves cross near floor [dB]: " + ", ".join(f"{f:.1f}" for f in crossings))
else:
    slower = "exact" if gap.min() > 0 else "minorized"
    print(f"no crossing: the {slower} filter needs more steps at every floor in that range "
          f"(by {abs(gap).min():.0f} to {abs(gap).max():.0f} steps)")

In [ ]:
SWEEP_SNR = [0.0, 5.0, 10.0, 15.0]

start = time.time()
sweep_b = {}                                   # (snr, filter) -> (epsilon, floor, steps, status, curve)
for snr in SWEEP_SNR:
    if snr == SNR_DB:                          # 5 dB is exactly the run of section 4: reuse it
        for name, epsilon, misalignment in [("sKF-L (minorized)", EPSILON_MIN, misalignment_min),
                                            ("sKF-L (exact)", EPSILON_L, misalignment_L)]:
            floor, reached = steady_state(misalignment)
            sweep_b[(snr, name)] = (epsilon, floor, reached, selected[name][3] + ", run of section 4", misalignment)
        continue

    # generate_signals and run_filter read var_eta, b_eta and scale_gg off the globals, so the
    # sweep reassigns them here and puts them back after the loop.
    var_eta = P_signal/10**(snr/10)
    b_eta = np.sqrt(var_eta/2)
    scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    search_signals = [generate_signals(seed)[:2] for seed in range(R_SEARCH)]
    plot_signals = [generate_signals(seed)[:2] for seed in range(R_PLOT)]

    for name in LAPLACIAN:
        floors, _ = scan_grid(name, EPS_GRID_LAPLACIAN, N_SEARCH, search_signals)
        sweep_b[(snr, name)] = pick_epsilon(name, EPS_GRID_LAPLACIAN, floors, TARGET_DB, N_SEARCH, plot_signals)

var_eta = P_signal/10**(SNR_DB/10)              # back to the scenario of section 2
b_eta = np.sqrt(var_eta/2)
scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
search_signals = [generate_signals(seed)[:2] for seed in range(R_SEARCH)]
plot_signals = [generate_signals(seed)[:2] for seed in range(R_PLOT)]

print(f"var_eta back to {var_eta:.4e}, the value section 2 printed")
print(f"phase 3, SNR sweep: {time.time() - start:.0f} s")

**Figure 4.** Steps to converge, exact over minorized, at equal floor, against SNR. Above the dashed line the minorized filter is faster.

In [ ]:
def speed_ratio(snr):
    """Steps of the exact filter over steps of the minorized one at one SNR; nan if either missed the target."""
    _, _, steps_exact, status_exact, _ = sweep_b[(snr, "sKF-L (exact)")]
    _, _, steps_min, status_min, _ = sweep_b[(snr, "sKF-L (minorized)")]
    if not (status_exact.startswith("ok") and status_min.startswith("ok")):
        return np.nan
    return steps_exact/steps_min


fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(SWEEP_SNR, [speed_ratio(snr) for snr in SWEEP_SNR], "o-")
ax.axhline(1.0, color="k", linestyle="--", linewidth=0.8)
ax.set_xticks([0, 5, 10, 15])                  # exactly the swept values
ax.set_xlabel("SNR [dB]")
ax.set_ylabel("steps, exact / minorized")
ax.set_title(f"Speed at equal floor, target {TARGET_DB:.0f} dB: above 1, the minorized filter is faster")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'SNR':>8}  {'ratio':>6}  {'floor, exact - minorized [dB]':>30}")
for snr in SWEEP_SNR:
    gap = sweep_b[(snr, "sKF-L (exact)")][1] - sweep_b[(snr, "sKF-L (minorized)")][1]
    print(f"{snr:>5.0f} dB  {speed_ratio(snr):>6.2f}  {gap:>30.2f}")

**Figure 5.** The two Laplacian filters at each SNR, with the $-20$ dB target dashed. The table below gives $\varepsilon$, floor and steps.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8, 15), sharey=True)
for ax, letter, snr in zip(axes, "abcd", SWEEP_SNR):
    slowest = 0
    for name, color in [("sKF-L (minorized)", "C1"), ("sKF-L (exact)", "C2")]:
        epsilon, floor, reached, status, misalignment = sweep_b[(snr, name)]
        ax.plot(np.arange(len(misalignment))/FS, 10*np.log10(misalignment), color=color,
                linewidth=1.2, label=f"{name}, eps = {epsilon:.1e}")
        slowest = max(slowest, reached)
    ax.axhline(TARGET_DB, color="k", linestyle="--", linewidth=0.8)
    ax.text(0.99, TARGET_DB, f"target = {TARGET_DB:.0f} dB", transform=ax.get_yaxis_transform(),
            ha="right", va="bottom", fontsize=8, bbox=dict(facecolor="white", edgecolor="none", pad=1))
    ax.set_xlim(0, min(3*slowest, len(misalignment))/FS)   # three times the slower convergence
    ax.set_title(f"({letter}) SNR = {snr:.0f} dB")
    ax.set_xlabel("$t$ [sec]")
    ax.set_ylabel("misalignment [dB]")
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)

# The y axis is shared, so one set of ticks serves all four panels: the default ones plus -20 dB.
low, high = axes[0].get_ylim()
axes[0].set_yticks(sorted({t for t in axes[0].get_yticks() if low <= t <= high} | {TARGET_DB}))
fig.suptitle(f"Convergence of the robust filters at target {TARGET_DB:.0f} dB")
plt.tight_layout(rect=(0, 0, 1, 0.98))     # room for the title
plt.show()

print(f"{'SNR':>5}  {'filter':<20}{'epsilon':>11}{'floor [dB]':>12}{'steps':>9}  status")
for snr in SWEEP_SNR:
    for name in LAPLACIAN:
        epsilon, floor, reached, status, _ = sweep_b[(snr, name)]
        eps_text = "-" if epsilon is None else f"{epsilon:.2e}"
        steps_text = "-" if reached is None else str(reached)
        print(f"{snr:>5.0f}  {name:<20}{eps_text:>11}{floor:>12.2f}{steps_text:>9}  {status}")

print(f"\nwhole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min")

## 7. Conclusion

**Gaussian against Laplacian: the Laplacian filters get there about nine times faster.** Section 5 prints the floors
(`floor_gauss`, `floor_min`, `floor_L`), the steps (`reached_gauss`, `reached_min`, `reached_L`) and the speed-ups; the
paper reports the same tenfold gap between its Figs. 3a and 3b, under the same conditions. At $R = 3$ the search estimates
the Gaussian floor at $-19.71$ dB and flags it as *target not reached*; at $R = 20$ it reaches $-20.17$ dB. The flag is an
artifact of the cheap search, but the filter is genuinely at the edge of what it can reach at this SNR.

**Minorized against exact, at SNR $= 5$ dB: the minorized filter's trade-off curve lies below the exact one's wherever the two can be compared.**
Figure 3 has no crossing over the common floor range, $-28.3$ to $-6.3$ dB: at every floor there, the exact filter needs more steps.
The gap grows with depth, from about 6% more steps at $-10$ dB to about 30% at $-20$ dB and 65% at $-28$ dB. At the shallow end,
$-8$ to $-6$ dB, the curves come within 2 to 3%, inside what $R = 10$ and interpolation between grid points can resolve: there they
effectively touch. Below $-28$ dB only the minorized filter has settled grid points, down to $-33$ dB. So at this SNR, over the scanned
range, the minorized filter dominates and no choice of $\varepsilon$ reverses the result. The starred points are one reading of the
same figure: at a common $\varepsilon$ the minorized filter sits 3.26 dB deeper for 9.6% more steps, and to reach that depth the exact
filter would need roughly 1.5 times as many steps.

**Across SNR, the advantage shrinks.** At equal floor the exact filter needs 1.63 times the minorized filter's steps at 0 dB and
1.03 times at 15 dB (Figure 4). The trade-off curve is measured only at 5 dB, so dominance is established there, not at the other SNRs.

This runs against expected outcome (iv) of the draft. **Why is still unexplained**; that is analytical work on the two
updates, not more simulation.

## 8. Limitations

* **Room fixed.** No $T_c$, no switch from $h^{(1)}$ to $h^{(2)}$.
* **Recovery speed not measured.** It is the main criterion of section 7 of the draft, and without a room change there is nothing to recover from.
* **Impulse response truncated** to $M = 128$ samples, 16 ms of a 200 ms tail. The draft specifies exactly this, and the data come from the truncated response, so nothing is mismatched.
* **Noise shape mismatched on purpose.** Neither filter is told $\beta^* = 0.2$. Laplacian noise was tried first: there the filters land within tenths of a dB of each other.
* **SNR is nominal.** With $\beta^* = 0.2$ the noise power of a finite run swings widely around $v_\eta$.
* **The search runs cheap.** $R = 3$ and shorter runs while scanning. An unconverged filter shows a higher floor and gets a larger $\varepsilon$; the Gaussian filter scans at full $N$ for that reason.
* **Finite grid.** Two points per decade. The selected $\varepsilon$ is interpolated and verified by one run, so its floor is reported, not assumed. On the trade-off curve the points are half a decade apart, and the crossing check interpolates linearly between them.
* **Equal floor is approximate.** The table under Figure 4 prints the floor gap per condition; where it reaches tenths of a dB, the filter that went deeper pays for it in steps.
* **Few realisations.** $R = 20$ for the run of section 4 and the starred points, $R = 10$ for the trade-off curve and the SNR sweep. With $\beta^* = 0.2$, floors are worth a few tenths of a dB.
* **$-25$ dB is not the common target**, because the Gaussian filter never reaches it at 5 dB SNR. The SNR sweep leaves that filter out, for cost.
* **`x_reg` dropped.** The closed forms of notebooks 01 and 02 regularised the input only to match Augusto's grid; there is no grid here.
* **Cost.** The exact filter is ~15x slower per step than the other two, because it works in the log domain: $\Phi$ underflows and $e^{2e_t/b_\eta}$ overflows. That sizes every search here.
* **The trade-off curve is at one SNR and $N = 48000$.** Grid points that did not settle within half that length are left out, as is the minorized filter's stalled branch; the table under Figure 3 lists them.
* **Three filters, one noise shape.** No fKF, SG or LF, and $\beta^*$ is not swept.

## 9. Observations on the imported code

`sKF_L_algorithm` is Ramiro's, copied unchanged; nothing below was fixed.
* Its two update lines match eqs. (50) and (51) of the draft term by term.
* `v_hist` is `(N, L)` but repeats one scalar across the columns: 12 MB per call that nothing reads.
* The weight history comes back under `h`; his `NLMS_algorithm` returns the final weights under the same key.
* Its floor is not monotonic in $\varepsilon$: below the optimum the filter freezes. That is why the search scans the whole grid.